In [34]:

from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from datasets import load_from_disk, DatasetDict
from qwen_vl_utils import process_vision_info
import os
import torch
import torch.nn.functional as F

import dotenv
dotenv.load_dotenv()

True

In [ ]:

REASONING_MODEL = 'Jakh0103/Qwen2.5-VL-3B-GRPO-VSR'

ORIGINAL_DATASET_PATH =  os.environ['DATA_PATH'] + "/vsr"
OUTPUT_DATASET_PATH =  os.environ['DATA_PATH'] + "/vsr_prompt_tuning"

# default: Load the model on the available device(s)
reasoning_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    REASONING_MODEL, torch_dtype="auto", device_map="cuda:0"
)

# default processor
processor = AutoProcessor.from_pretrained(REASONING_MODEL, use_fast=True)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
# preprocess the dataset
dataset = load_from_disk(ORIGINAL_DATASET_PATH)

In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['caption', 'label', 'relation', 'subj', 'obj', 'image_path'],
        num_rows: 3489
    })
    validation: Dataset({
        features: ['caption', 'label', 'relation', 'subj', 'obj', 'image_path'],
        num_rows: 340
    })
    test: Dataset({
        features: ['caption', 'label', 'relation', 'subj', 'obj', 'image_path'],
        num_rows: 1222
    })
})

In [ ]:
NUM_SAMPLES = {'train': 150, 'validation': 10, 'test': 10}
dataset = {k: v.select(range(NUM_SAMPLES[k])) for k, v in dataset.items()}



In [ ]:
dataset = {
    k: di.map(
        lambda sample: {
            "problem": f'Is the following statement true: {sample["caption"]}',
            "solution": str(sample["label"] == 1),
        },
        remove_columns=["caption", "label", "relation", "subj", "obj"],
        desc="Preprocessing dataset",
    )
    for k, di in dataset.items()
}



In [ ]:
def make_conversation_from_prompt(example, prompt_template, answer = None):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": f"file://{example['image_path']}"},
                {"type": "text", "text": prompt_template(example)},
            ],
        }
    ]
    if answer is not None:
        messages.append(
            {
                "role": "assistant",
                "content": [
                    {"type": "text", "text": answer},
                ],
            }
        )
    return {"messages": messages, "solution": example['solution']}

# apply formatting
def reasoning_prompt_template(example):
    return f"{example['problem']} First output the thinking process in <think> </think> tags and then output the final answer in <answer> </answer> tags."

def baseline_prompt_template(example):
    return f"{example['problem']} First output the thinking process in <think> </think> tags and then output the final answer in <answer> </answer> tags."


In [ ]:
TH_TOKEN_ID = processor.tokenizer.convert_tokens_to_ids("<th")

def add_reasoning_output(item):
    conversation = make_conversation_from_prompt(item, reasoning_prompt_template)

    text = processor.apply_chat_template(
        conversation['messages'], tokenize=False, add_generation_prompt=True
    )

    image_inputs, video_inputs = process_vision_info(conversation['messages'])

    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )
    inputs = inputs.to(reasoning_model.device)


    # Inference: Generation of the output
    with torch.no_grad():
        # Generate the output
        generated_ids = reasoning_model.generate(**inputs, max_new_tokens=256)
        generated_ids_trimmed = [
            out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        output_text_batch = processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )
        assert len(output_text_batch) == 1
        output_text = output_text_batch[0]
    
    training_conversation = make_conversation_from_prompt(item, baseline_prompt_template, output_text)
    training_text = processor.apply_chat_template(
        training_conversation['messages'], tokenize=False, add_generation_prompt=False
    )
    training_inputs = processor(
        text=[training_text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )
    reasoning_token_indexes = (training_inputs.input_ids[0] == TH_TOKEN_ID).nonzero().flatten().tolist()
    assert len(reasoning_token_indexes) == 1
    reasoning_offset = reasoning_token_indexes[0]

    with torch.no_grad():
        output = reasoning_model(
            **training_inputs.to(reasoning_model.device),
            return_dict=True,
        )





    return {
        "desired_output": output_text,
        "training_inputs": training_inputs,
        "reasoning_logits_offset": reasoning_offset,
        "reasoning_logits": output.logits[0, reasoning_offset:],
    }

r = add_reasoning_output(dataset['train'][0])

{'desired_output': '<think>\nThe image shows a close-up of a chocolate cake with candles and colorful candies on top, placed on a table. There is no visible person in the frame, so it cannot be determined if the cake is next to someone.\n</think>\n<answer>\nTrue\n</answer>',
 'training_inputs': {'input_ids': tensor([[151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
          151645,    198, 151644,    872,    198, 151652, 151655, 151655, 151655,
          151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
          151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
          151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
          151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
          151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
          151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
          151655, 151655, 151655, 1

In [ ]:
tokenizer = processor.tokenizer
tokenizer.tokenize("<th", )
th_token_id = tokenizer.convert_tokens_to_ids("<think")
th_token_id = tokenizer.convert_tokens_to_ids("<think")

13708

In [ ]:

dataset = {k: v.map(add_reasoning_output) for k, v in dataset.items()}


In [ ]:
dataset

{'train': Dataset({
     features: ['image_path', 'problem', 'solution', 'desired_output'],
     num_rows: 1000
 }),
 'validation': Dataset({
     features: ['image_path', 'problem', 'solution', 'desired_output'],
     num_rows: 10
 }),
 'test': Dataset({
     features: ['image_path', 'problem', 'solution', 'desired_output'],
     num_rows: 10
 })}

In [ ]:
DatasetDict(dataset).save_to_disk(OUTPUT_DATASET_PATH)

Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/10 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/10 [00:00<?, ? examples/s]

In [ ]:
train = dataset['train']
item = train[0]
item['desired_output']

'<think>\nThe image shows a close-up of a chocolate cake with candles and colorful candies on top, placed on a table. There is no visible person in the frame, so it cannot be determined if the cake is next to someone.\n</think>\n<answer>\nTrue\n</answer>'

: 